### Clone the repo and install dependencies

In [ ]:
%cd /content
!git clone https://github.com/Comfy-Org/ComfyUI.git
%cd /content/ComfyUI
!pip install -r requirements.txt
!git clone https://github.com/Comfy-Org/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
!pip install -r custom_nodes/ComfyUI-Manager/requirements.txt


### Connect Google Drive to store downloaded models and load them

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')

folders = (
    'checkpoints',
    'loras',
    'vae',
    'diffusion_models',
    'text_encoders',
    'clip',
    'background_removal',
    'clip_vision',
    'geometry_estimation',
)

for folder in folders:
    (MODELS_DIR / folder).mkdir(parents=True, exist_ok=True)

# Add the Drive config without removing other settings.
config = Path('/content/ComfyUI/extra_model_paths.yaml')
existing = config.read_text() if config.exists() else ''
additions = []

if 'drive_models:' not in existing:
    additions.append("""drive_models:
  base_path: /content/drive/MyDrive/ComfyUI/models
  checkpoints: checkpoints
  loras: loras
  vae: vae
  diffusion_models: diffusion_models
  text_encoders: text_encoders
  clip: clip""")

if 'drive_3d:' not in existing:
    additions.append("""drive_3d:
  base_path: /content/drive/MyDrive/ComfyUI/models
  background_removal: background_removal
  clip_vision: clip_vision
  geometry_estimation: geometry_estimation""")

if additions:
    separator = '\n\n' if existing.strip() else ''
    config.write_text(existing.rstrip() + separator + '\n\n'.join(additions) + '\n')
    print('Drive config added.')
else:
    print('Drive config already present.')

print('Drive ready:', MODELS_DIR)



### Download VAE, text encoder, diffusion model, etc. To Drive

In [ ]:
# DISCLAIMER (Hugging Face links):
# Hugging Face links usually look like `/blob/main/`, which opens the repo's overview/file page.
# Before downloading or running this cell, change them to `/resolve/main/` so the link goes
# straight to the file download instead of the overview tab.
from pathlib import Path
import requests
from tqdm.auto import tqdm

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')
ASSETS = [
    # Add your assets here in the format:
    ('your_folder', 'your_filename', 'your_url'),
    # You can add multiple assets as needed. For example:
    # ('diffusion_models', 'example_model.safetensors', 'https://huggingface.co/example_user/example_model/resolve/main/example_model.safetensors'),
    # ('loras', 'example_lora.safetensors', 'https://huggingface.co/example_user/example_lora/resolve/main/example_lora.safetensors'),
    # ('vae', 'example_vae.vae.pt', 'https://huggingface.co/example_user/example_vae/resolve/main/example_vae.vae.pt'),
]

session = requests.Session()
prepared = []
print('Checking links and existing files...')
for folder, filename, url in ASSETS:
    destination = MODELS_DIR / folder / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    response = session.head(url, allow_redirects=True, timeout=60)
    response.raise_for_status()
    expected = int(response.headers.get('Content-Length', 0))
    if destination.exists() and expected and destination.stat().st_size == expected:
        print(f'Already complete, skipping: {filename}')
        continue
    if destination.exists() and not expected and destination.stat().st_size > 0:
        print(f'File already exists (server size unavailable), skipping: {filename}')
        continue
    prepared.append((destination, url, expected, filename))

if not prepared:
    print('All assets are already available on Drive.')
else:
    total = sum(size for _, _, size, _ in prepared)
    print(f'{len(prepared)} file(s) to download; detected total size: {total / (1024**3):.2f} GiB.')
    for destination, url, expected, filename in prepared:
        partial = destination.with_name(destination.name + '.part')
        partial.unlink(missing_ok=True)
        try:
            with session.get(url, stream=True, allow_redirects=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                with partial.open('wb') as output, tqdm(
                    total=expected or None, unit='B', unit_scale=True, unit_divisor=1024, desc=filename
                ) as progress:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            output.write(chunk)
                            progress.update(len(chunk))
            downloaded = partial.stat().st_size
            if expected and downloaded != expected:
                raise IOError(f'Size mismatch for {filename}: got {downloaded} of {expected} bytes.')
            partial.replace(destination)
            print(f'Done: {destination}')
        except Exception:
            partial.unlink(missing_ok=True)
            raise

print('Done. If ComfyUI is already running, refresh the model list or restart the server so the models are detected.')

### Download a checkpoint or LoRA to Google Drive

In [ ]:
# Fill in a Civitai or Hugging Face URL, then run this cell.
#
# DISCLAIMER (Civitai downloads):
# Set up a Civitai API key as a Colab secret named 'CIVITAI_API_TOKEN'
# (Colab left sidebar > Secrets > enable notebook access).
# It is only used so the download is not rejected with a 401 error.
URL = 'https://civitai.com/models/modelVersionId'
# or 'https://huggingface.co/example_user/example_model/resolve/main/example_model.safetensors'
# For Hugging Face files in the repo root, choose the target folder here.
HF_MODEL_TYPE = 'loras'

from pathlib import Path
from urllib.parse import urlparse, parse_qs
from google.colab import userdata
import requests

MODELS_DIR = Path('/content/drive/MyDrive/ComfyUI/models')
parsed = urlparse(URL.strip())
host = (parsed.hostname or '').lower()
if host.startswith('www.'):
    host = host[4:]
parts = parsed.path.strip('/').split('/')
query = parse_qs(parsed.query)

if host == 'huggingface.co':
    if len(parts) < 5 or parts[2] not in ('blob', 'resolve'):
        raise ValueError('Enter a Hugging Face file link in /blob/ or /resolve/ format.')
    owner, repo, _, revision = parts[:4]
    file_path = parts[4:]
    if not file_path:
        raise ValueError('The Hugging Face link does not point to a file.')
    download_url = f'https://huggingface.co/{owner}/{repo}/resolve/{revision}/' + '/'.join(file_path)
    allowed_types = {'checkpoints', 'loras', 'vae', 'diffusion_models', 'text_encoders', 'clip'}
    if file_path[0] in allowed_types:
        kind = file_path[0]
        relative_parts = file_path
    else:
        kind = HF_MODEL_TYPE.strip().lower()
        relative_parts = [file_path[-1]]
    if kind not in allowed_types:
        raise ValueError(f'HF_MODEL_TYPE must be one of: {sorted(allowed_types)}')
    destination = MODELS_DIR / kind
    for part in relative_parts:
        if part in ('', '.', '..'):
            raise ValueError('Invalid file path.')
        destination = destination / part
    headers = {}
elif host in ('civitai.com', 'civitai.red'):
    token = None
    try:
        token = userdata.get('CIVITAI_API_TOKEN')
    except Exception:
        pass
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    api_base = f'https://{host}'
    version_id = query.get('modelVersionId', [None])[0]
    file_id = query.get('fileId', [None])[0]
    is_download_link = len(parts) >= 4 and parts[-3:-1] == ['download', 'models']

    if is_download_link:
        model_id = parts[-1]
        response = requests.get(f'{api_base}/api/v1/models/{model_id}', headers=headers, timeout=30)
        response.raise_for_status()
        model_info = response.json()
        chosen = None
        for version in model_info.get('modelVersions') or []:
            for file in version.get('files') or []:
                if file_id and str(file.get('id')) == str(file_id):
                    chosen = (version, file)
                    break
            if chosen:
                break
        if chosen is None:
            version = (model_info.get('modelVersions') or [{}])[0]
            files = version.get('files') or []
            selected_file = next((f for f in files if f.get('primary')), files[0] if files else None)
            if selected_file:
                chosen = (version, selected_file)
        if not chosen:
            raise RuntimeError('Civitai did not return any model file.')
        version_info, selected_file = chosen
        download_url = URL.strip()
    else:
        if not version_id:
            raise ValueError('The Civitai link must have a modelVersionId or be a /api/download/models/ link.')
        response = requests.get(f'{api_base}/api/v1/model-versions/{version_id}', headers=headers, timeout=30)
        response.raise_for_status()
        version_info = response.json()
        model_info = version_info.get('model') or {}
        files = version_info.get('files') or []
        selected_file = next((f for f in files if file_id and str(f.get('id')) == str(file_id)), None)
        selected_file = selected_file or next((f for f in files if f.get('primary')), files[0] if files else None)
        if not selected_file:
            raise RuntimeError('Civitai did not return any file for this version.')
        model_id = model_info.get('id')
        download_url = selected_file.get('downloadUrl') or f'{api_base}/api/download/models/{model_id}?fileId={selected_file.get("id")}'

    kind_map = {'lora': 'loras', 'checkpoint': 'checkpoints', 'textualinversion': 'embeddings', 'vae': 'vae', 'controlnet': 'controlnet'}
    kind = kind_map.get((model_info.get('type') or '').lower(), 'loras')
    filename = selected_file.get('name') or 'model.safetensors'
    destination = MODELS_DIR / kind / filename
    headers['Authorization'] = f'Bearer {token}' if token else ''
else:
    raise ValueError('Unsupported host. Use civitai.com, civitai.red, or huggingface.co.')

if headers.get('Authorization') == '':
    headers.pop('Authorization')
destination.parent.mkdir(parents=True, exist_ok=True)
if destination.exists():
    print(f'Already exists, skipping: {destination} ({destination.stat().st_size:,} bytes)')
else:
    partial = destination.with_name(destination.name + '.part')
    if partial.exists():
        raise FileExistsError(f'Temporary file already exists and will not be overwritten: {partial}')
    print(f'Downloading {destination.name} to {destination}', flush=True)
    try:
        with requests.get(download_url, headers=headers, allow_redirects=True, stream=True, timeout=(60, 180)) as response:
            response.raise_for_status()
            expected = int(response.headers.get('Content-Length', 0)) or None
            downloaded = 0
            with open(partial, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
            if expected is not None and downloaded != expected:
                raise IOError(f'Download size mismatch: got {downloaded:,} of {expected:,} bytes.')
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
    print(f'Saved: {destination} ({destination.stat().st_size:,} bytes)')
print('If ComfyUI is already running, refresh the model list; restart the server only if the model does not show up.')

### Run ComfyUI with cloudflared (Recommended)




In [ ]:
from pathlib import Path
import getpass
import hashlib
import os
import re
import secrets
import shutil
import socket
import subprocess
import time
import urllib.request

COMFY_DIR = Path("/content/ComfyUI")
COMFY_LOG = Path("/content/comfyui.log")
PROXY_LOG = Path("/content/comfy_auth_proxy.log")
TUNNEL_LOG = Path("/content/cloudflared.log")
PROXY_SCRIPT = Path("/tmp/comfy_auth_proxy.py")
TOKEN_FILE = Path("/tmp/.comfy_auth_hash")

COMFY_PORT = 8188   # ComfyUI internal port (local 127.0.0.1 only)
AUTH_PORT = 8189    # Password-protected reverse proxy port (exposed to cloudflared)
# Public hostname served by the named tunnel. Kept in a Colab secret (`COMFY_DOMAIN`)
# or env var so this file carries no personal domain.
COMFY_DOMAIN = ""
try:
    from google.colab import userdata
    COMFY_DOMAIN = userdata.get("COMFY_DOMAIN") or ""
except Exception:
    COMFY_DOMAIN = ""
COMFY_DOMAIN = (COMFY_DOMAIN or os.environ.get("COMFY_DOMAIN") or "").rstrip("/")
DOMAIN_LABEL = COMFY_DOMAIN or "(named tunnel active, set the COMFY_DOMAIN secret to display its hostname)"

# Colab's network stalls badly on QUIC/UDP (each cold request adds ~2s of retransmit
# backoff). Forcing HTTP/2 over TCP keeps first-byte latency at ~0.2s. Do not remove.
TUNNEL_PROTOCOL = "http2"

def port_is_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            return True
    except OSError:
        return False

# 1. Get password from Colab Secret `COMFY_PASSWORD`, or prompt for hidden input (getpass).
password = None
try:
    from google.colab import userdata
    password = userdata.get("COMFY_PASSWORD")
except Exception:
    password = None

if password:
    print("Using password from Colab Secret `COMFY_PASSWORD`.", flush=True)
    auth_token = hashlib.sha256(f"comfy-salt:{password}".encode()).hexdigest()
    TOKEN_FILE.write_text(auth_token)
elif TOKEN_FILE.exists() and port_is_open(AUTH_PORT):
    auth_token = TOKEN_FILE.read_text().strip()
    print("Using active session password previously set.", flush=True)
else:
    entered = getpass.getpass("Enter password to lock ComfyUI Tunnel (leave empty to auto-generate): ").strip()
    if not entered:
        entered = secrets.token_urlsafe(12)
        print(f"Password automatically generated (save this, will not show on web): {entered}", flush=True)
    auth_token = hashlib.sha256(f"comfy-salt:{entered}".encode()).hexdigest()
    TOKEN_FILE.write_text(auth_token)
    del entered
del password

# 2. Install cloudflared only if the fresh runtime does not have it yet.
if shutil.which("cloudflared") is None:
    deb_path = "/tmp/cloudflared-linux-amd64.deb"
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        deb_path,
    )
    subprocess.run(["dpkg", "-i", deb_path], check=True)

if not COMFY_DIR.exists():
    raise FileNotFoundError(f"ComfyUI folder not found: {COMFY_DIR}")

# 3. Stop old ComfyUI instance if running without --enable-cors-header flag (causes 403 on splash screen).
for cmdline in Path("/proc").glob("[0-9]*/cmdline"):
    try:
        args = cmdline.read_bytes().replace(b"\0", b" ").decode(errors="ignore")
        if "python" in args and "main.py" in args and "--enable-cors-header" not in args:
            pid = int(cmdline.parent.name)
            os.kill(pid, 9)
            print(f"Stopping old ComfyUI instance without CORS flag (PID {pid}).", flush=True)
            time.sleep(1)
    except OSError:
        pass

# 4. Run ComfyUI with the --enable-cors-header flag.
comfyui_process = None
if port_is_open(COMFY_PORT):
    print(f"ComfyUI is already active on internal port {COMFY_PORT}; not launching a duplicate.", flush=True)
else:
    comfy_log_file = COMFY_LOG.open("w")
    comfyui_process = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT), "--enable-cors-header", "--dont-print-server"],
        cwd=COMFY_DIR,
        stdout=comfy_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    comfy_log_file.close()
    print(f"ComfyUI launched in the background (PID {comfyui_process.pid}); waiting for port {COMFY_PORT}...", flush=True)

    deadline = time.time() + 180
    while time.time() < deadline and not port_is_open(COMFY_PORT):
        if comfyui_process.poll() is not None:
            tail = COMFY_LOG.read_text(errors="replace")[-5000:]
            raise RuntimeError("ComfyUI stopped during startup; see log " + str(COMFY_LOG) + " | " + tail)
        time.sleep(2)
    if not port_is_open(COMFY_PORT):
        raise TimeoutError(f"ComfyUI was not ready after 180 seconds. Check {COMFY_LOG}")

# 5. Run Auth Reverse Proxy (aiohttp) on port 8189 -> 8188.
PROXY_SCRIPT.write_text('''import asyncio
import hashlib
import hmac
import os
import time
from pathlib import Path
from aiohttp import web, ClientSession, ClientTimeout, WSMsgType

UPSTREAM_HTTP = f"http://127.0.0.1:{os.environ['COMFY_PORT']}"
UPSTREAM_WS = f"ws://127.0.0.1:{os.environ['COMFY_PORT']}"
AUTH_PORT = int(os.environ["AUTH_PORT"])
TOKEN_FILE = Path("/tmp/.comfy_auth_hash")
COOKIE_NAME = "comfy_colab_session"

failed_attempts = {}

LOGIN_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>ComfyUI Private Access</title>
<style>
  body { background:#121212; color:#e0e0e0; font-family:system-ui,-apple-system,sans-serif; display:flex; align-items:center; justify-content:center; height:100vh; margin:0; }
  .card { background:#1e1e1e; padding:2rem; border-radius:12px; border:1px solid #333; width:100%; max-width:340px; box-shadow:0 8px 24px rgba(0,0,0,0.5); }
  h2 { margin:0 0 0.5rem; font-size:1.25rem; }
  p { margin:0 0 1.25rem; color:#999; font-size:0.9rem; }
  input { width:100%; box-sizing:border-box; padding:0.7rem 0.85rem; border-radius:8px; border:1px solid #444; background:#141414; color:#fff; font-size:0.95rem; margin-bottom:1rem; }
  input:focus { outline:none; border-color:#4f8cff; }
  button { width:100%; padding:0.7rem; border:none; border-radius:8px; background:#2563eb; color:#fff; font-weight:600; font-size:0.95rem; cursor:pointer; }
  button:hover { background:#1d4ed8; }
  .err { color:#f87171; font-size:0.85rem; margin-bottom:1rem; }
</style>
</head>
<body>
  <form class="card" method="POST" action="/__comfy_login">
    <h2>ComfyUI Locked</h2>
    <p>Enter password to access this instance.</p>
    __ERR__
    <input type="password" name="password" placeholder="Password" autofocus required>
    <button type="submit">Login</button>
  </form>
</body>
</html>"""

def get_expected_token():
    return TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""

def is_authenticated(request):
    cookie_val = request.cookies.get(COOKIE_NAME, "")
    expected = get_expected_token()
    return bool(expected) and hmac.compare_digest(cookie_val, expected)

async def handle_login(request):
    ip = request.headers.get("CF-Connecting-IP", request.remote or "unknown")
    now = time.time()
    attempts = [t for t in failed_attempts.get(ip, []) if now - t < 60]
    failed_attempts[ip] = attempts
    if len(attempts) >= 5:
        return web.Response(
            text=LOGIN_HTML.replace("__ERR__", '<div class="err">Too many failed attempts. Please wait 1 minute.</div>'),
            content_type="text/html",
            status=429,
        )

    data = await request.post()
    pwd = str(data.get("password", "")).strip()
    candidate = hashlib.sha256(f"comfy-salt:{pwd}".encode()).hexdigest()
    expected = get_expected_token()

    if expected and hmac.compare_digest(candidate, expected):
        failed_attempts.pop(ip, None)
        resp = web.HTTPFound("/")
        resp.set_cookie(
            COOKIE_NAME,
            expected,
            max_age=86400,
            httponly=True,
            secure=True,
            samesite="Lax",
        )
        return resp

    attempts.append(now)
    return web.Response(
        text=LOGIN_HTML.replace("__ERR__", '<div class="err">Incorrect password.</div>'),
        content_type="text/html",
        status=401,
    )

async def proxy_handler(request):
    if request.path == "/__comfy_login" and request.method == "POST":
        return await handle_login(request)

    if not is_authenticated(request):
        if request.headers.get("Upgrade", "").lower() == "websocket" or request.path.startswith("/api/") or request.path in ("/prompt", "/queue", "/history"):
            return web.Response(status=401, text="Unauthorized")
        return web.Response(
            text=LOGIN_HTML.replace("__ERR__", ""),
            content_type="text/html",
            status=401,
        )

    session = request.app["client_session"]

    # Proxy WebSocket (/ws)
    if request.headers.get("Upgrade", "").lower() == "websocket":
        ws_server = web.WebSocketResponse(max_msg_size=0)
        await ws_server.prepare(request)
        target_ws_url = f"{UPSTREAM_WS}{request.rel_url}"

        async with session.ws_connect(target_ws_url, max_msg_size=0) as ws_client:
            async def forward(src, dst):
                try:
                    async for msg in src:
                        if msg.type == WSMsgType.TEXT:
                            await dst.send_str(msg.data)
                        elif msg.type == WSMsgType.BINARY:
                            await dst.send_bytes(msg.data)
                        elif msg.type == WSMsgType.CLOSE:
                            await dst.close()
                finally:
                    if not dst.closed:
                        await dst.close()

            await asyncio.gather(forward(ws_server, ws_client), forward(ws_client, ws_server), return_exceptions=True)
        return ws_server

    # Regular HTTP proxy
    target_url = f"{UPSTREAM_HTTP}{request.rel_url}"
    skip_req_headers = {"content-length", "transfer-encoding", "connection", "upgrade"}
    req_headers = [(k, v) for k, v in request.headers.items() if k.lower() not in skip_req_headers]
    body = await request.read()

    async with session.request(request.method, target_url, headers=req_headers, data=body, allow_redirects=False) as upstream_resp:
        resp_body = await upstream_resp.read()
        skip_resp_headers = {"content-length", "transfer-encoding", "connection", "keep-alive"}
        # Hashed build assets never change -> let the browser keep them forever,
        # so repeat visits skip the tunnel entirely for JS/CSS/fonts.
        is_asset = request.path.startswith("/assets/") or request.path.endswith((".js", ".css", ".woff", ".woff2"))
        if is_asset:
            skip_resp_headers.add("cache-control")
        resp_headers = [(k, v) for k, v in upstream_resp.headers.items() if k.lower() not in skip_resp_headers]
        if is_asset:
            resp_headers.append(("Cache-Control", "public, max-age=31536000, immutable"))
        return web.Response(status=upstream_resp.status, headers=resp_headers, body=resp_body)

async def on_startup(app):
    app["client_session"] = ClientSession(auto_decompress=False, timeout=ClientTimeout(total=None))

async def on_cleanup(app):
    await app["client_session"].close()

app = web.Application(client_max_size=1024**3)
app.on_startup.append(on_startup)
app.on_cleanup.append(on_cleanup)
app.router.add_route("*", "/{tail:.*}", proxy_handler)

if __name__ == "__main__":
    web.run_app(app, host="127.0.0.1", port=AUTH_PORT, print=None)
''')

if not port_is_open(AUTH_PORT):
    proxy_env = os.environ.copy()
    proxy_env["COMFY_PORT"] = str(COMFY_PORT)
    proxy_env["AUTH_PORT"] = str(AUTH_PORT)
    proxy_log_file = PROXY_LOG.open("w")
    proxy_process = subprocess.Popen(
        ["python", str(PROXY_SCRIPT)],
        env=proxy_env,
        stdout=proxy_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    proxy_log_file.close()
    for _ in range(20):
        if port_is_open(AUTH_PORT):
            break
        time.sleep(0.5)
    if not port_is_open(AUTH_PORT):
        raise RuntimeError(f"Auth proxy failed to run on port {AUTH_PORT}. Check {PROXY_LOG}")
    print(f"Auth Proxy active on port {AUTH_PORT} (PID {proxy_process.pid}).", flush=True)
else:
    print(f"Auth Proxy is already active on port {AUTH_PORT}.", flush=True)

# 6. Resolve the named-tunnel token (Colab Secret first, then env var). Empty -> temporary quick tunnel.
CF_TOKEN = None
try:
    from google.colab import userdata
    CF_TOKEN = userdata.get("COMFY_TUNNEL_TOKEN") or userdata.get("CLOUDFLARE_TUNNEL_TOKEN")
except Exception:
    CF_TOKEN = None
CF_TOKEN = (CF_TOKEN or os.environ.get("COMFY_TUNNEL_TOKEN") or os.environ.get("CLOUDFLARE_TUNNEL_TOKEN") or "").strip()

USE_NAMED = bool(CF_TOKEN)

def tunnel_matches_mode(args):
    """True only for a tunnel that fits the mode selected right now."""
    if f"--protocol {TUNNEL_PROTOCOL}" not in args:
        return False
    if USE_NAMED:
        return "--token" in args
    return "--token" not in args and f"127.0.0.1:{AUTH_PORT}" in args

# Keep at most one tunnel, and only if it matches the current mode. Anything else
# is killed: a leftover named tunnel keeps serving the custom domain even when the
# token secret is off, which silently runs two tunnels against the same proxy.
active_tunnel = False
for cmdline in Path("/proc").glob("[0-9]*/cmdline"):
    try:
        args = cmdline.read_bytes().replace(bytes([0]), b" ").decode(errors="ignore")
        if "cloudflared" not in args or "tunnel" not in args:
            continue
        if tunnel_matches_mode(args) and not active_tunnel:
            active_tunnel = True
            print("Keeping existing", "named" if USE_NAMED else "quick", "tunnel.", flush=True)
            continue
        pid = int(cmdline.parent.name)
        os.kill(pid, 9)
        kind = "named" if "--token" in args else "quick"
        print(f"Killed stale {kind} tunnel (PID {pid}).", flush=True)
        time.sleep(1)
    except OSError:
        pass

# 7. Run Cloudflare Tunnel to AUTH_PORT (8189): named tunnel on the custom domain, else quick tunnel.
cloudflared_process = None
if active_tunnel:
    print("Protected Cloudflare Tunnel is already active.", flush=True)
    if USE_NAMED:
        print("ComfyUI URL (Password Protected):", DOMAIN_LABEL, flush=True)
    else:
        log_text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            print("ComfyUI URL (Password Protected):", match.group(0), flush=True)
else:
    if USE_NAMED:
        print("Using COMFY_TUNNEL_TOKEN for the named tunnel:", DOMAIN_LABEL, flush=True)
        tunnel_cmd = ["cloudflared", "tunnel", "--no-autoupdate", "--protocol", TUNNEL_PROTOCOL, "run", "--token", CF_TOKEN]
    else:
        print("No COMFY_TUNNEL_TOKEN found; falling back to temporary Quick Tunnel.", flush=True)
        tunnel_cmd = ["cloudflared", "tunnel", "--protocol", TUNNEL_PROTOCOL, "--url", f"http://127.0.0.1:{AUTH_PORT}", "--no-autoupdate"]

    tunnel_log_file = TUNNEL_LOG.open("w")
    cloudflared_process = subprocess.Popen(
        tunnel_cmd,
        stdout=tunnel_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    tunnel_log_file.close()
    print(f"Cloudflare Tunnel launched in the background (PID {cloudflared_process.pid}).", flush=True)

    deadline = time.time() + 30
    tunnel_url = None
    while time.time() < deadline:
        if cloudflared_process.poll() is not None:
            break
        log_text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
        if USE_NAMED:
            # A named tunnel never prints a URL; wait for the connector handshake instead.
            if "Registered tunnel connection" in log_text:
                tunnel_url = COMFY_DOMAIN
                break
        else:
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
            if match:
                tunnel_url = match.group(0)
                break
        time.sleep(1)
    if tunnel_url:
        print("ComfyUI URL (Password Protected):", tunnel_url, flush=True)
    else:
        print(f"Tunnel is still starting or has not provided a URL yet. Check log: {TUNNEL_LOG}", flush=True)

if USE_NAMED:
    print("Note: the custom domain is only served while this named tunnel is running.", flush=True)
else:
    print("Note: COMFY_TUNNEL_TOKEN is off, so the custom domain is not served right now.", flush=True)

print(f"ComfyUI log: {COMFY_LOG}")
print(f"Auth Proxy log: {PROXY_LOG}")
print(f"Tunnel log: {TUNNEL_LOG}")


In [ ]:
# This is optional, you don't need to run this cell in order to open the web UI.
# Its only for proxying the ComfyUI port through Colab's JS interface, which is not always reliable.
from google.colab import output
print(output.eval_js("google.colab.kernel.proxyPort(8188)"))

### Run ComfyUI with localtunnel




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

### Run ComfyUI with a Colab iframe (fallback in case localtunnel fails)

If you see a 403 error, try opening the link in another browser.

Some UI features such as live image preview may not work because the Colab iframe blocks websockets.

In [ ]:
import os, socket, subprocess, time
from google.colab import output

log = open('/tmp/comfyui.log', 'w')
server = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--dont-print-server'],
    cwd='/content/ComfyUI', stdout=log, stderr=subprocess.STDOUT, start_new_session=True
)
for _ in range(120):
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        if server.poll() is not None:
            log.flush()
            print(open('/tmp/comfyui.log').read()[-12000:])
            raise RuntimeError('ComfyUI stopped before port 8188 was ready')
        time.sleep(1)
else:
    print(open('/tmp/comfyui.log').read()[-12000:])
    raise TimeoutError('ComfyUI did not open port 8188 within 120 seconds')
output.serve_kernel_port_as_iframe(8188, height=1024)

In [ ]:
import requests
r = requests.get('http://127.0.0.1:8188/', timeout=10)
print('HTTP', r.status_code, 'bytes', len(r.content))
print(r.text[:300])

In [ ]:
import requests
urls = [
  'https://civitai.red/api/download/models/3218603?fileId=3100615',
  'https://civitai.red/api/download/models/2575595?fileId=2463002',
]
for url in urls:
    r = requests.get(url, allow_redirects=True, stream=True, timeout=45)
    print('status:', r.status_code, 'final host:', r.url.split('/')[2])
    print('type:', r.headers.get('Content-Type'), 'size:', r.headers.get('Content-Length'), 'name:', r.headers.get('Content-Disposition'))
    r.close()

In [ ]:
import os, requests
print('DRIVE_MOUNTED:', os.path.ismount('/content/drive'), 'MYDRIVE_EXISTS:', os.path.isdir('/content/drive/MyDrive'))
for version in (3218603, 2575595):
    r = requests.get(f'https://civitai.com/api/v1/model-versions/{version}', timeout=30)
    print('VERSION', version, 'HTTP', r.status_code)
    if r.ok:
        data = r.json()
        print('name:', data.get('name'), '| base:', data.get('baseModel'))
        for f in data.get('files', []):
            print('file:', f.get('name'), '| type:', f.get('type'), '| bytes:', f.get('sizeKB'), 'KB | primary:', f.get('primary'))
    else:
        print(r.text[:300])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('MOUNTED:', __import__('os').path.isdir('/content/drive/MyDrive'))

In [ ]:
from google.colab import userdata
from pathlib import Path
import requests, shutil, os, time
token=userdata.get('CIVITAI_API_TOKEN')
if not token: raise RuntimeError('Secret CIVITAI_API_TOKEN is not available')
items=[
    (3218603,3100615,'oneObsession_v24.safetensors','checkpoints',6938040682),
    (2575595,2463002,'AIイラストおじさん_V3-Exp-Compact.safetensors','loras',912618100),
]
for mid,fid,name,kind,expected in items:
    local_dir=Path('/content/ComfyUI/models')/kind
    drive_dir=Path('/content/drive/MyDrive/ComfyUI/models')/kind
    local_dir.mkdir(parents=True,exist_ok=True); drive_dir.mkdir(parents=True,exist_ok=True)
    dest=local_dir/name; drive_dest=drive_dir/name
    if dest.exists() or drive_dest.exists(): raise FileExistsError(f'Destination file already exists: {name}; not overwriting')
    part=local_dir/(name+'.part'); drive_part=drive_dir/(name+'.part')
    if part.exists() or drive_part.exists(): raise FileExistsError(f'.part file already exists: {name}; not overwriting')
    url=f'https://civitai.com/api/download/models/{mid}?fileId={fid}'
    print(f'Starting {name} ({expected:,} bytes)',flush=True)
    with requests.get(url,headers={'Authorization':f'Bearer {token}'},allow_redirects=True,stream=True,timeout=(60,180)) as r:
        if r.status_code != 200: raise RuntimeError(f'Download of {name} failed: HTTP {r.status_code}; {r.text[:200]}')
        length=r.headers.get('Content-Length')
        if length and int(length)!=expected: raise RuntimeError(f'Server size does not match for {name}: {length}')
        downloaded=0; report=0; started=time.time()
        with open(part,'wb') as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if chunk:
                    f.write(chunk); downloaded+=len(chunk)
                    if downloaded-report>=256*1024*1024:
                        print(f'  {name}: {downloaded/expected:.0%} ({downloaded:,}/{expected:,} bytes)',flush=True); report=downloaded
            f.flush(); os.fsync(f.fileno())
    if downloaded!=expected: raise RuntimeError(f'Download size mismatch for {name}: got {downloaded:,} of {expected:,} bytes')
    part.replace(dest)
    print(f'Local download finished: {dest.stat().st_size:,} bytes; copying to Drive...',flush=True)
    shutil.copyfile(dest,drive_part)
    if drive_part.stat().st_size!=expected: raise RuntimeError(f'Drive copy size mismatch for {name}')
    drive_part.replace(drive_dest)
    print(f'SAVED: {drive_dest} ({drive_dest.stat().st_size:,} bytes)',flush=True)
del token
print('DONE: both files saved to Google Drive.',flush=True)

### Last resort

In [ ]:
import socket
import subprocess
import time
from google.colab import output

def comfyui_running():
    try:
        with socket.create_connection(("127.0.0.1", 8188), timeout=2):
            return True
    except OSError:
        return False

if not comfyui_running():
    log = open("/content/comfyui.log", "a")
    process = subprocess.Popen(
        ["python", "main.py", "--dont-print-server"],
        cwd="/content/ComfyUI",
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log.close()

    for _ in range(180):
        if comfyui_running():
            break
        time.sleep(1)

if not comfyui_running():
    raise RuntimeError("ComfyUI is not running. Check the log at /content/comfyui.log")

url = output.eval_js("google.colab.kernel.proxyPort(8188)")
print("ComfyUI URL:", url)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8188, height=900)

### Fuse musubi Qwen-Image 2.1 LoRA into ComfyUI's fused `gate_up` form

musubi-tuner splits the SwiGLU MLP into two Linears (`img_mlp.gate_layer` + `img_mlp.proj`), while ComfyUI packs both into one GEMM (`img_mlp.gate_up`). A LoRA trained on the split layout therefore loses 64 modules (2 per block x 32 blocks) at inference, which shows up as:

```
Model QwenImage21 prepared ... 160 patches attached      <- should be 224
lora key not loaded: lora_unet_transformer_blocks_0_img_mlp_proj.lora_down.weight
```

This cell finds the native checkpoints on Drive by itself, rewrites each block's `gate_layer` + `proj` deltas into a single rank-2r delta on `gate_up`, and emits the rest unchanged in diffusers format. The fusion is asserted numerically before writing.

Run it, then reload the ComfyUI tab (F5) so the `lora_name` dropdown refreshes, and pick the `_fused` file.

Expected per file: `modules in: 224 | gate_up fused: 32 | leftover old img_mlp keys: 0 | unmapped: 0`


In [ ]:
# Fuse musubi Qwen-Image 2.1 LoRA (img_mlp.gate_layer + img_mlp.proj) into ComfyUI img_mlp.gate_up
import re
from pathlib import Path

import torch
from safetensors.torch import load_file, save_file

DRIVE_ROOT = Path("/content/drive/MyDrive")
DST_DIR = DRIVE_ROOT / "ComfyUI/models/loras"

# Directories that are known to hold the trained checkpoints
CANDIDATE_DIRS = [
    DRIVE_ROOT / "Qwen_Image_Character_LoRA_Outputs",
    DRIVE_ROOT / "LoRA_Outputs",
    DRIVE_ROOT / "ComfyUI/models/loras",
    DRIVE_ROOT / "Qwen_Image_Base_Models",
]

MODULE_PATTERNS = [
    "time_text_embed.timestep_embedder.linear_1",
    "time_text_embed.timestep_embedder.linear_2",
    "txt_norm",
    "img_in",
    "txt_in",
    "transformer_blocks.*.img_mod.1",
    "transformer_blocks.*.attn.norm_q",
    "transformer_blocks.*.attn.norm_k",
    "transformer_blocks.*.attn.to_q",
    "transformer_blocks.*.attn.to_k",
    "transformer_blocks.*.attn.to_v",
    "transformer_blocks.*.attn.add_k_proj",
    "transformer_blocks.*.attn.add_v_proj",
    "transformer_blocks.*.attn.add_q_proj",
    "transformer_blocks.*.attn.to_out.0",
    "transformer_blocks.*.attn.to_add_out",
    "transformer_blocks.*.attn.norm_added_q",
    "transformer_blocks.*.attn.norm_added_k",
    "transformer_blocks.*.img_mlp.out",
    "transformer_blocks.*.txt_mod.1",
    "transformer_blocks.*.txt_mlp.net.0.proj",
    "transformer_blocks.*.txt_mlp.net.2",
    "norm_out.linear",
    "proj_out",
]

PREFIX = "lora_unet_"
BLOCK_RE = re.compile(re.escape(PREFIX) + r"transformer_blocks_(\d+)_img_mlp_(gate_layer|proj)$")

# lora_unet_transformer_blocks_0_attn_to_q -> transformer_blocks.0.attn.to_q
LOOKUP = {}
for pattern in MODULE_PATTERNS:
    if "*" in pattern:
        for i in range(200):
            key = (PREFIX + pattern.replace("*", str(i))).replace(".", "_")
            LOOKUP[key] = pattern.replace("*", str(i))
    else:
        LOOKUP[PREFIX + pattern.replace(".", "_")] = pattern


def discover_sources():
    """Find the native musubi checkpoints on Drive instead of assuming one path."""
    found = []
    for directory in CANDIDATE_DIRS:
        if not directory.exists():
            print(f"  (not found) {directory}")
            continue
        files = sorted(f for f in directory.glob("*.safetensors") if _is_source(f))
        print(f"  {directory}  ->  {len(files)} candidate file(s)")
        found.extend(files)

    if not found:
        print("\n  Fallback: scanning MyDrive two levels deep...")
        for top in sorted(x for x in DRIVE_ROOT.iterdir() if x.is_dir()):
            try:
                levels = [top] + sorted(y for y in top.iterdir() if y.is_dir())
            except OSError:
                levels = [top]
            for directory in levels:
                try:
                    for f in directory.glob("*.safetensors"):
                        if _is_source(f):
                            found.append(f)
                except OSError:
                    pass

    if not found:
        print("\n  No matching checkpoint found. Top level of MyDrive:")
        for top in sorted(DRIVE_ROOT.iterdir())[:60]:
            print("    ", "DIR " if top.is_dir() else "file", top.name)
    return found


def _is_source(path):
    name = path.name.lower()
    if not name.endswith(".safetensors"):
        return False
    if name.endswith("_fused.safetensors") or name.endswith("_comfy.safetensors"):
        return False
    return "elsa" in name or "qwen_image_21" in name


def group_tensors(sd):
    """Split flat state dict into {module_name: {down, up, alpha}}."""
    groups = {}
    for key, tensor in sd.items():
        if not key.startswith(PREFIX):
            continue
        for suffix, part in ((".lora_down.weight", "down"), (".lora_up.weight", "up"), (".alpha", "alpha")):
            if key.endswith(suffix):
                groups.setdefault(key[: -len(suffix)], {})[part] = tensor
                break
    return groups


def fuse_file(src_path, dst_path):
    groups = group_tensors(load_file(str(src_path)))

    blocks, others = {}, []
    for module, parts in groups.items():
        match = BLOCK_RE.match(module)
        if match:
            blocks.setdefault(int(match.group(1)), {})[match.group(2)] = parts
        else:
            others.append((module, parts))

    out, unmapped, fused_count = {}, [], 0

    # 1. every non-MLP module, alpha/rank factor baked into lora_B
    for module, parts in others:
        path = LOOKUP.get(module)
        if path is None or "down" not in parts or "up" not in parts:
            unmapped.append(module)
            continue
        rank = parts["down"].shape[0]
        scale = float(parts["alpha"]) / rank if "alpha" in parts else 1.0
        out[f"diffusion_model.{path}.lora_A.weight"] = parts["down"].float()
        out[f"diffusion_model.{path}.lora_B.weight"] = parts["up"].float() * scale

    # 2. fuse gate_layer + proj per block into one rank-2r delta on gate_up
    for index, pair in sorted(blocks.items()):
        if "gate_layer" not in pair or "proj" not in pair:
            unmapped.append(f"block {index} incomplete pair")
            continue

        gate, up = pair["gate_layer"], pair["proj"]
        a_gate, b_gate = gate["down"].float(), gate["up"].float()
        a_up, b_up = up["down"].float(), up["up"].float()

        r_gate, r_up = a_gate.shape[0], a_up.shape[0]
        s_gate = float(gate["alpha"]) / r_gate if "alpha" in gate else 1.0
        s_up = float(up["alpha"]) / r_up if "alpha" in up else 1.0

        hidden = b_gate.shape[0]
        down = torch.cat([a_gate, a_up], dim=0)
        fused_up = torch.zeros(2 * hidden, r_gate + r_up, dtype=torch.float32)
        fused_up[:hidden, :r_gate] = b_gate * s_gate
        fused_up[hidden:, r_gate:] = b_up * s_up

        # the fused delta must equal [gate_delta ; up_delta] stacked vertically
        reference = torch.cat([(b_gate * s_gate) @ a_gate, (b_up * s_up) @ a_up], dim=0)
        assert torch.allclose(reference, fused_up @ down, atol=1e-4, rtol=1e-4), f"fusion mismatch in block {index}"

        out[f"diffusion_model.transformer_blocks.{index}.img_mlp.gate_up.lora_A.weight"] = down
        out[f"diffusion_model.transformer_blocks.{index}.img_mlp.gate_up.lora_B.weight"] = fused_up
        fused_count += 1

    out = {k: v.contiguous() for k, v in out.items()}
    save_file(out, str(dst_path), metadata={"format": "pt", "converted_by": "fuse_gate_up"})

    leftover = [k for k in out if "img_mlp.gate_layer" in k or "img_mlp.proj" in k]
    print(f"   {src_path.name}")
    print(f"      modules in: {len(groups):3d} | gate_up fused: {fused_count:3d} | tensors written: {len(out):4d}")
    print(f"      leftover old img_mlp keys: {len(leftover)} | unmapped: {len(unmapped)}")
    for module in unmapped[:10]:
        print("         unmapped:", module)
    return leftover, unmapped


if not Path("/content/drive/MyDrive").exists():
    raise RuntimeError("Google Drive is not mounted in this runtime")

print("Looking for checkpoints:")
sources = discover_sources()
print()

if not sources:
    print("Nothing to convert. Fix the path above, then run this cell again.")
else:
    DST_DIR.mkdir(parents=True, exist_ok=True)
    total_bad = 0
    for source in sources:
        target = DST_DIR / f"{source.stem}_fused.safetensors"
        leftover, unmapped = fuse_file(source, target)
        total_bad += len(leftover) + len(unmapped)
        print(f"      -> {target}\n")

    print("=" * 70)
    if total_bad == 0:
        print("All good. Reload the ComfyUI tab (F5), then pick the *_fused.safetensors files.")
    else:
        print(f"{total_bad} problem(s) above - send the unmapped key names.")
